# Inverse Rollout: Sensitivity Analysis

This notebook demonstrates how to use the inverse rollout functionality to compute sensitivities of model outputs with respect to initial conditions. This is useful for understanding which parts of the initial state most strongly influence a particular output variable at a specific location and time.

**Use case**: You want to know which initial condition perturbations would most efficiently cause a temperature decrease at a specific grid point over several forecast steps.

## Prerequisites

This notebook requires the same data as the ERA5 example. Please run `example_era5.ipynb` first to download the required data, or ensure you have ERA5 data available.

```
pip install cdsapi matplotlib
```

## Load the Data

We'll use the same ERA5 data preparation as in the standard example.

In [ ]:
from pathlib import Path

import torch
import xarray as xr

from aurora import Batch, Metadata

# Path where ERA5 data was downloaded
download_path = Path("~/downloads").expanduser()

# Load the datasets
static_vars_ds = xr.open_dataset(download_path / "static.nc", engine="netcdf4")
surf_vars_ds = xr.open_dataset(download_path / "2023-01-01-surface-level.nc", engine="netcdf4")
atmos_vars_ds = xr.open_dataset(download_path / "2023-01-01-atmospheric.nc", engine="netcdf4")

# Create the batch
batch = Batch(
    surf_vars={
        "2t": torch.from_numpy(surf_vars_ds["t2m"].values[:2][None]),
        "10u": torch.from_numpy(surf_vars_ds["u10"].values[:2][None]),
        "10v": torch.from_numpy(surf_vars_ds["v10"].values[:2][None]),
        "msl": torch.from_numpy(surf_vars_ds["msl"].values[:2][None]),
    },
    static_vars={
        "z": torch.from_numpy(static_vars_ds["z"].values[0]),
        "slt": torch.from_numpy(static_vars_ds["slt"].values[0]),
        "lsm": torch.from_numpy(static_vars_ds["lsm"].values[0]),
    },
    atmos_vars={
        "t": torch.from_numpy(atmos_vars_ds["t"].values[:2][None]),
        "u": torch.from_numpy(atmos_vars_ds["u"].values[:2][None]),
        "v": torch.from_numpy(atmos_vars_ds["v"].values[:2][None]),
        "q": torch.from_numpy(atmos_vars_ds["q"].values[:2][None]),
        "z": torch.from_numpy(atmos_vars_ds["z"].values[:2][None]),
    },
    metadata=Metadata(
        lat=torch.from_numpy(surf_vars_ds.latitude.values),
        lon=torch.from_numpy(surf_vars_ds.longitude.values),
        time=(surf_vars_ds.valid_time.values.astype("datetime64[s]").tolist()[1],),
        atmos_levels=tuple(int(level) for level in atmos_vars_ds.pressure_level.values),
    ),
)

print(f"Batch spatial shape: {batch.spatial_shape}")
print(f"Latitude range: {batch.metadata.lat.min():.1f} to {batch.metadata.lat.max():.1f}")
print(f"Longitude range: {batch.metadata.lon.min():.1f} to {batch.metadata.lon.max():.1f}")

## Load the Model

We load Aurora and configure it for gradient computation. For memory efficiency with gradient computation, we enable activation checkpointing.

In [ ]:
from aurora import Aurora

model = Aurora(use_lora=False)  # The pretrained version does not use LoRA
model.load_checkpoint("microsoft/aurora", "aurora-0.25-pretrained.ckpt")

# Enable activation checkpointing to reduce memory usage during backpropagation
model.configure_activation_checkpointing()

model.eval()
model = model.to("cuda")

print("Model loaded successfully!")

## Define the Target Point

We'll compute sensitivities for 2m temperature at a specific location. Let's choose a point over central Europe.

In [ ]:
import numpy as np

# Target location: approximately central Europe (50°N, 10°E)
target_lat = 50.0
target_lon = 10.0

# Find the nearest grid indices
lat_idx = int(torch.argmin(torch.abs(batch.metadata.lat - target_lat)).item())
lon_idx = int(torch.argmin(torch.abs(batch.metadata.lon - target_lon)).item())

actual_lat = batch.metadata.lat[lat_idx].item()
actual_lon = batch.metadata.lon[lon_idx].item()

print(f"Target location: {target_lat}°N, {target_lon}°E")
print(f"Nearest grid point: {actual_lat}°N, {actual_lon}°E")
print(f"Grid indices: lat_idx={lat_idx}, lon_idx={lon_idx}")

## Compute Sensitivities

Now we compute the sensitivity of the temperature trajectory at our target point with respect to the initial conditions. We define a perturbation direction that represents wanting to *decrease* the temperature.

In [ ]:
from aurora import (
    compute_initial_perturbation,
    create_trajectory_perturbation_loss,
)

# Number of rollout steps
steps = 4

# Define the perturbation direction: we want to find sensitivities for DECREASING temperature
# A direction of -1 means: "which initial changes would decrease this output?"
perturbation_direction = torch.tensor([-1.0] * steps)

# Create the loss function
loss_fn = create_trajectory_perturbation_loss(
    var_name="2t",  # 2m temperature
    lat_idx=lat_idx,
    lon_idx=lon_idx,
    perturbation_direction=perturbation_direction,
    var_type="surf",
)

print(f"Computing sensitivities over {steps} rollout steps...")
print("This may take a minute due to backpropagation through the model.")

# Compute the gradients
result = compute_initial_perturbation(
    model=model,
    initial_batch=batch,
    steps=steps,
    loss_fn=loss_fn,
    return_predictions=True,
)

print(f"\nComputation complete!")
print(f"Loss value: {result['loss']:.4f}")

## Visualize the Sensitivity Maps

The gradients tell us which initial condition changes would most effectively decrease temperature at our target point. Let's visualize the sensitivity of the 2m temperature output to the initial 2m temperature field.

In [ ]:
import matplotlib.pyplot as plt

# Get the gradient for initial 2m temperature
# Shape: (batch, time_history, height, width)
temp_grad = result["surf_var_grads"]["2t"]

print(f"Temperature gradient shape: {temp_grad.shape}")
print(f"  - Batch dimension: {temp_grad.shape[0]}")
print(f"  - History timesteps: {temp_grad.shape[1]}")
print(f"  - Spatial: {temp_grad.shape[2]} x {temp_grad.shape[3]}")

# Plot sensitivity maps for both history timesteps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for t in range(2):
    grad_map = temp_grad[0, t].numpy()
    
    # Use symmetric colormap centered at 0
    vmax = np.abs(grad_map).max()
    
    im = axes[t].imshow(
        grad_map,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        extent=[0, 360, -90, 90],
        origin="upper",
    )
    axes[t].plot(actual_lon, actual_lat, "k*", markersize=15, label="Target point")
    axes[t].set_title(f"Sensitivity to T at t-{1-t} (history step {t})")
    axes[t].set_xlabel("Longitude")
    axes[t].set_ylabel("Latitude")
    axes[t].legend()
    plt.colorbar(im, ax=axes[t], label="dLoss/dT (K⁻¹)")

plt.suptitle(
    f"Sensitivity of temperature at ({actual_lat}°N, {actual_lon}°E) to initial temperature",
    fontsize=12,
)
plt.tight_layout()
plt.show()

## Interpret the Results

The sensitivity maps show:
- **Red regions**: Increasing temperature here would *increase* temperature at the target (positive gradient with our negative perturbation direction means decreasing input would decrease output)
- **Blue regions**: Increasing temperature here would *decrease* temperature at the target

The strongest sensitivities are typically:
1. Near the target point itself (local effects)
2. Upstream in the flow direction (advection)
3. In regions that influence large-scale patterns affecting the target

In [ ]:
# Zoom in on the region around the target point
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Define zoom region (±30 degrees around target)
lat_min, lat_max = max(-90, actual_lat - 30), min(90, actual_lat + 30)
lon_min, lon_max = max(0, actual_lon - 40), min(360, actual_lon + 40)

# Find corresponding indices
lat_mask = (batch.metadata.lat >= lat_min) & (batch.metadata.lat <= lat_max)
lon_mask = (batch.metadata.lon >= lon_min) & (batch.metadata.lon <= lon_max)

lat_indices = torch.where(lat_mask)[0]
lon_indices = torch.where(lon_mask)[0]

for t in range(2):
    # Extract zoomed region
    grad_zoomed = temp_grad[0, t, lat_indices[0]:lat_indices[-1]+1, lon_indices[0]:lon_indices[-1]+1].numpy()
    
    vmax = np.abs(grad_zoomed).max()
    
    im = axes[t].imshow(
        grad_zoomed,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        extent=[lon_min, lon_max, lat_min, lat_max],
        origin="upper",
    )
    axes[t].plot(actual_lon, actual_lat, "k*", markersize=15, label="Target point")
    axes[t].set_title(f"Zoomed sensitivity (history step {t})")
    axes[t].set_xlabel("Longitude")
    axes[t].set_ylabel("Latitude")
    axes[t].legend()
    plt.colorbar(im, ax=axes[t], label="dLoss/dT (K⁻¹)")

plt.suptitle("Zoomed view of temperature sensitivity", fontsize=12)
plt.tight_layout()
plt.show()

## Sensitivities to Other Variables

We can also examine how the target temperature is sensitive to other initial variables, such as mean sea level pressure or atmospheric temperature.

In [ ]:
# Plot sensitivities to different initial variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

variables = [
    ("2t", "surf", "2m Temperature", "K⁻¹"),
    ("msl", "surf", "Mean Sea Level Pressure", "Pa⁻¹"),
    ("10u", "surf", "10m U-Wind", "(m/s)⁻¹"),
    ("10v", "surf", "10m V-Wind", "(m/s)⁻¹"),
]

for idx, (var_name, var_type, title, unit) in enumerate(variables):
    ax = axes[idx // 2, idx % 2]
    
    if var_type == "surf":
        grad = result["surf_var_grads"][var_name]
    else:
        grad = result["atmos_var_grads"][var_name]
    
    if grad is None:
        ax.text(0.5, 0.5, "No gradient computed", ha="center", va="center")
        ax.set_title(title)
        continue
    
    # Use the most recent history timestep
    if var_type == "surf":
        grad_map = grad[0, -1].numpy()
    else:
        # For atmospheric, sum over levels or pick one
        grad_map = grad[0, -1].sum(dim=0).numpy()
    
    vmax = np.abs(grad_map).max()
    if vmax == 0:
        vmax = 1  # Avoid division by zero
    
    im = ax.imshow(
        grad_map,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        extent=[0, 360, -90, 90],
        origin="upper",
    )
    ax.plot(actual_lon, actual_lat, "k*", markersize=10)
    ax.set_title(f"Sensitivity to {title}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, label=f"dLoss/d{var_name} ({unit})")

plt.suptitle(
    f"Sensitivity of temperature at ({actual_lat}°N, {actual_lon}°E) to various initial fields",
    fontsize=12,
)
plt.tight_layout()
plt.show()

## Sensitivity to Atmospheric Variables

Let's also look at how the surface temperature is influenced by the atmospheric temperature at different pressure levels.

In [ ]:
# Get atmospheric temperature gradient
atmos_t_grad = result["atmos_var_grads"]["t"]
print(f"Atmospheric temperature gradient shape: {atmos_t_grad.shape}")
print(f"Pressure levels: {batch.metadata.atmos_levels}")

# Plot sensitivity at different pressure levels
levels_to_plot = [1000, 850, 500, 200]  # hPa
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, level in enumerate(levels_to_plot):
    ax = axes[idx // 2, idx % 2]
    
    # Find the level index
    level_idx = batch.metadata.atmos_levels.index(level)
    
    # Extract gradient for this level (most recent history step)
    grad_map = atmos_t_grad[0, -1, level_idx].numpy()
    
    vmax = np.abs(grad_map).max()
    if vmax == 0:
        vmax = 1
    
    im = ax.imshow(
        grad_map,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        extent=[0, 360, -90, 90],
        origin="upper",
    )
    ax.plot(actual_lon, actual_lat, "k*", markersize=10)
    ax.set_title(f"Sensitivity to T at {level} hPa")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, label="dLoss/dT (K⁻¹)")

plt.suptitle(
    f"Sensitivity of surface T at ({actual_lat}°N, {actual_lon}°E) to atmospheric T",
    fontsize=12,
)
plt.tight_layout()
plt.show()

## Extract the Predicted Trajectory

We can also examine the actual predicted temperature trajectory at our target point.

In [ ]:
from aurora import extract_timeseries

# Extract the temperature time series from the predictions
predictions = result["predictions"]
temp_timeseries = extract_timeseries(
    predictions,
    var_name="2t",
    lat_idx=lat_idx,
    lon_idx=lon_idx,
    var_type="surf",
)

# Convert to Celsius
temp_celsius = temp_timeseries.detach().cpu().numpy() - 273.15

# Get forecast times
base_time = batch.metadata.time[0]
forecast_hours = [6 * (i + 1) for i in range(steps)]

plt.figure(figsize=(10, 5))
plt.plot(forecast_hours, temp_celsius, "bo-", linewidth=2, markersize=8)
plt.xlabel("Forecast Lead Time (hours)")
plt.ylabel("Temperature (°C)")
plt.title(f"Predicted Temperature at ({actual_lat}°N, {actual_lon}°E)")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Temperature trajectory: {temp_celsius}")

## Multi-Point Sensitivity Analysis

You can also compute sensitivities for multiple points simultaneously using `create_multipoint_loss`.

In [ ]:
from aurora import create_multipoint_loss

# Define multiple target points
target_points = [
    {"name": "Central Europe", "lat": 50.0, "lon": 10.0},
    {"name": "UK", "lat": 52.0, "lon": 0.0},
    {"name": "Scandinavia", "lat": 60.0, "lon": 15.0},
]

# Build the points list for the loss function
points = []
for pt in target_points:
    lat_idx = int(torch.argmin(torch.abs(batch.metadata.lat - pt["lat"])).item())
    lon_idx = int(torch.argmin(torch.abs(batch.metadata.lon - pt["lon"])).item())
    
    points.append({
        "var_name": "2t",
        "lat_idx": lat_idx,
        "lon_idx": lon_idx,
        "perturbation_direction": torch.tensor([-1.0] * steps),
        "var_type": "surf",
        "weight": 1.0,
    })
    print(f"{pt['name']}: lat_idx={lat_idx}, lon_idx={lon_idx}")

# Create multi-point loss
multipoint_loss_fn = create_multipoint_loss(points)

print("\nComputing multi-point sensitivities...")

multipoint_result = compute_initial_perturbation(
    model=model,
    initial_batch=batch,
    steps=steps,
    loss_fn=multipoint_loss_fn,
)

print(f"Multi-point loss: {multipoint_result['loss']:.4f}")

In [ ]:
# Visualize multi-point sensitivity
fig, ax = plt.subplots(figsize=(12, 6))

grad_map = multipoint_result["surf_var_grads"]["2t"][0, -1].numpy()
vmax = np.abs(grad_map).max()

im = ax.imshow(
    grad_map,
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    extent=[0, 360, -90, 90],
    origin="upper",
)

# Mark all target points
for pt in target_points:
    ax.plot(pt["lon"], pt["lat"], "k*", markersize=12)
    ax.annotate(pt["name"], (pt["lon"] + 2, pt["lat"] + 2), fontsize=10)

ax.set_title("Combined sensitivity for multiple European locations")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.colorbar(im, ax=ax, label="dLoss/dT (K⁻¹)")
plt.tight_layout()
plt.show()

## Cleanup

In [ ]:
# Move model back to CPU to free GPU memory
model = model.to("cpu")
torch.cuda.empty_cache()
print("Cleanup complete!")

## Summary

In this notebook, we demonstrated how to:

1. **Compute sensitivities** of forecast outputs to initial conditions using `compute_initial_perturbation`
2. **Define perturbation directions** using `create_trajectory_perturbation_loss`
3. **Visualize sensitivity maps** showing which initial condition changes most affect the target
4. **Analyze sensitivities** across different variables and pressure levels
5. **Compute multi-point sensitivities** using `create_multipoint_loss`

This approach is useful for:
- Understanding model dynamics and error propagation
- Identifying key regions for observation targeting
- Ensemble perturbation design
- Data assimilation applications